# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [1]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
try:
    from google.colab import auth
except ImportError:
    auth = None
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")


✅ All libraries are ready to go!


### Authenticate and Configure Your Project
To use Vertex AI, you need an active Google Cloud project. This section handles authenticating your environment and setting up the necessary project configurations.

In [2]:
# ---  Authentication & Project Configuration ---

# Authenticate user in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Authenticated successfully.")

In [3]:
# @title Set Your Google Cloud Project Details
PROJECT_ID = "adk1-496817"             # @param {type:"string"}
LOCATION = "us-central1"               # @param {type:"string"}

if "google.colab" in sys.modules:
    # Use Vertex AI when running in Colab with an authenticated Google Cloud project
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
    !gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}
    print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")
else:
    # Outside Colab, use the Gemini API with GOOGLE_API_KEY from the environment
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
    assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your environment."
    print("\n✅ Gemini API configured via GOOGLE_API_KEY.")


✅ Gemini API configured via GOOGLE_API_KEY.


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [4]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-3.1-flash-lite",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [5]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    # Gemini quotas are enforced per minute, so retry transient 429 / 503 responses.
    for attempt in range(6):
        transient_error = None
        try:
            async for event in runner.run_async(
                user_id=user_id,
                session_id=session.id,
                new_message=Content(parts=[Part(text=query)], role="user")
            ):
                error_message = getattr(event, "error_message", None) or ""
                if "RESOURCE_EXHAUSTED" in error_message or "UNAVAILABLE" in error_message:
                    transient_error = error_message
                    continue
                if not is_router:
                    # Let's see what the agent is thinking!
                    print(f"EVENT: {event}")
                if event.is_final_response() and event.content:
                    final_response = event.content.parts[0].text
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) or "UNAVAILABLE" in str(e):
                transient_error = str(e)
            else:
                final_response = f"An error occurred: {e}"

        if final_response or transient_error is None:
            break
        print(f"⏳ Rate limited by the API, retrying in 45s (attempt {attempt + 1}/6)...")
        await asyncio.sleep(45)

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [6]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '57ec811a-1f77-44ab-adb3-b83f0cb2c6e4'...


Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


Node execution failed with exception
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asynci

Root node day_trip_agent failed.
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asyncio/__

⏳ Rate limited by the API, retrying in 45s (attempt 1/6)...


...

Node execution failed with exception
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asynci

Root node day_trip_agent failed.
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asyncio/__

⏳ Rate limited by the API, retrying in 45s (attempt 6/6)...



--------------------------------------------------
✅ Final Response:


--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [7]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-3.1-flash-lite",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [8]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: '6263b9da-7cc0-4838-b323-8b8eca5fe2e8'...


/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/llm_request.py:298: UserWarning: [EXPERIMENTAL] feature JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


Node execution failed with exception
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/workflow/_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/workflow/_node_runner.py", line 274, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/workflow/_node_runner.py", line 288, in _run_node_loop
    async for event in agen:
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/workflow/_base_node.py", line 170, in run
    async for item in agen:
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/agents/llm_agent.py", line 627, in _run_impl
    async for event in agen:
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/workflow/_llm_agent_wrapper.py", line 484, in run_llm_agent_as_node
    async for event in run_iter:


Root node weather_aware_planner failed.
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/runners.py", line 748, in _drive_root_node
    await root_ctx._run_node_internal(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/agents/context.py", line 608, in _run_node_internal
    raise DynamicNodeFailError(
google.adk.workflow._errors.DynamicNodeFailError: Dynamic node weather_aware_planner failed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/runners.py", line 1061, in _cleanup_root_task
    await task
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/runners.py", line 757, in _drive_root_node
    raise e.error
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/workflow/_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "/h

⏳ Rate limited by the API, retrying in 45s (attempt 1/6)...


...

Root node weather_aware_planner failed.
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/runners.py", line 748, in _drive_root_node
    await root_ctx._run_node_internal(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/agents/context.py", line 608, in _run_node_internal
    raise DynamicNodeFailError(
google.adk.workflow._errors.DynamicNodeFailError: Dynamic node weather_aware_planner failed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/runners.py", line 1061, in _cleanup_root_task
    await task
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/runners.py", line 757, in _drive_root_node
    raise e.error
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/workflow/_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "/h

⏳ Rate limited by the API, retrying in 45s (attempt 2/6)...


EVENT: model_version='gemini-3.1-flash-lite' content=Content(
  parts=[
    Part(
      text="""The current weather near Lake Tahoe is mostly sunny with a high of 63°F. There is a southwest wind blowing at around 10 mph, with gusts reaching up to 25 mph.

Because of those gusty winds, if you do head out for a hike, I recommend choosing a trail that is somewhat sheltered by trees to avoid feeling the full impact of the wind. Also, be sure to bring layers since 63°F can feel quite cool, especially when the wind picks up. Enjoy your hike!""",
      thought_signature=b'\x12q\no\x01\x11M2\x0f\xb4\xd5^\xa2\xf3\xd7\x0c\xac?T\xf3\xf1w\t;@\x9d`\xca\xf6q\x17u\xd8\xcf\xfeO\x11Y\xf2:Iy\xf9\x06\xeb\x07A\rj\xb6O~\r<\xb8\xc8iQY\xcc}\xcf\xd48\xdb\x90\xf2\xc8\x80\x95\xff`\xac\xe8\xc0>\x16\x19\x91\x0c\x8d\xc2\x1b5\xdalN\x8fL\x0e\x92\xbb)X\xd2\xa9\x8c...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None interaction_status=None finish_reason=<F

The current weather near Lake Tahoe is mostly sunny with a high of 63°F. There is a southwest wind blowing at around 10 mph, with gusts reaching up to 25 mph.

Because of those gusty winds, if you do head out for a hike, I recommend choosing a trail that is somewhat sheltered by trees to avoid feeling the full impact of the wind. Also, be sure to bring layers since 63°F can feel quite cool, especially when the wind picks up. Enjoy your hike!

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [9]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-3.1-flash-lite",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-3.1-flash-lite",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-3.1-flash-lite",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-3.1-flash-lite",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [10]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: '89d6be7f-3272-49cb-b59a-091e5efccddb'...


EVENT: model_version='gemini-3.1-flash-lite' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'What are the top-rated hotels in San Francisco?'
        },
        id='call_1624098',
        name='call_db_agent'
      ),
      thought_signature=b'\x12q\no\x01\x11M2\x0f\x86r\xaf\xbdL0\xfdr~\x89bw\x8f\xf2\x9d\x8c\x0e]iz\x93\x91\xd5\xa7\xf1\xe68k\xb6\x1d/\xe6\x1d>e).*8ICh\x95\xa4\x19\x18.N\xe0\xf5\x00$L9pB\xb2\xd4\xf9w\xcbn9^\x8e\xb6q!c\\\xb7\x06D\x9eE\\W\x88z`\xe7u\xc0X\x87\x06\x8fE...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None interaction_status=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=28,
  prompt_token_count=330,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,

EVENT: model_version=None content=Content(
  parts=[
    Part(
      function_response=FunctionResponse(
        id='call_1624098',
        name='call_db_agent',
        response={
          'result': "{'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}"
        }
      )
    ),
  ],
  role='user'
) grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None interaction_status=None finish_reason=None error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=None live_session_id=None go_away=None voice_activity=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None environment_id=None invocation_id='e-5fda5416-d47e-4f68-9b65-b196e1bf6de0' author='trip_data_concierge' actions=EventActions(skip_summarization=No

EVENT: model_version='gemini-3.1-flash-lite' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'The Seaside Inn in San Francisco has the most reviews (620). Can you suggest a good dinner spot near there?'
        },
        id='call_1114876',
        name='call_concierge_agent'
      ),
      thought_signature=b'\x12q\no\x01\x11M2\x0f\xe3\x86\xf2\xe725\xf8\x19\x16\xeb\xd1V\x1a\xa1^\xce\xd7\x8da\xbe\xcc\x84~\x04c\xf5\xd6\x84Z\xee\x06\xf2\x93\xd6\xf0\xd5k\x06k\x89G\xcd\x95\x97\xf02\xd3\xc7\r|\xbf[f\xee\xe8b\xf7\x16q-\x01\x10\xd9\xa9\xb4\xa0z\\\xe7\xa9+\xb3\x8b\xec\xb9\xe2}c\xd0\xcaR\xfcA\xc5o\x18\xcb&...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None interaction_status=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=43,

Node execution failed with exception
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/workflow/_node_runner.py", line 136, in run
    await self._execute_node(ctx, node_input)
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/workflow/_node_runner.py", line 274, in _execute_node
    await self._run_node_loop(ctx, node_input)
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/workflow/_node_runner.py", line 288, in _run_node_loop
    async for event in agen:
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/workflow/_base_node.py", line 170, in run
    async for item in agen:
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/agents/llm_agent.py", line 627, in _run_impl
    async for event in agen:
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/workflow/_llm_agent_wrapper.py", line 484, in run_llm_agent_as_node
    async for event in run_iter:


...

EVENT: model_version='gemini-3.1-flash-lite' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'The Seaside Inn has the most reviews (620) among the hotels in San Francisco. Can you suggest a good dinner spot near the Seaside Inn?'
        },
        id='call_673001',
        name='call_concierge_agent'
      ),
      thought_signature=b'\x12q\no\x01\x11M2\x0f\xb7{XCo\x8a\x98m\xe4\tW"\xdb\xc7\x93\x00`$\x86\xf2\xaa\xe3\xf5$\xaa\x05\xd0\x9b\xd8\xf7\x1f\xba:\x11\xe6\xd9\x924\x1c~+\xe0\xdd\xdb\x90o\xbb\x15\xeb\x11P\x1c\x92\xa6G|\x80LcN\xaf-\x9a\x02)\xff8J\x15\xa1c#f\xf8\xe2\xf8\xf7\x8d\x1b\x89s\xd7q$]De\x96...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None interaction_status=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count

EVENT: model_version=None content=Content(
  parts=[
    Part(
      function_response=FunctionResponse(
        id='call_673001',
        name='call_concierge_agent',
        response={
          'result': """Certainly. While I understand you are inquiring about the vicinity of the Seaside Inn, please allow me to share an expert perspective on the local dining scene.

Should you find yourself in that area, the food critic suggests **Outerlands**. They note that it is the only establishment in that part of the city capable of crafting a sourdough that is truly worthwhile. 

I hope this recommendation serves you well for your dinner plans. Is there anything else I may assist you with today?"""
        }
      )
    ),
  ],
  role='user'
) grounding_metadata=None partial=None turn_complete=None turn_complete_reason=None interaction_status=None finish_reason=None error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=Non

EVENT: model_version='gemini-3.1-flash-lite' content=Content(
  parts=[
    Part(
      text="""The top-rated hotels in San Francisco found in our database are **The Grand Hotel** (5 stars, 450 reviews) and the **Seaside Inn** (4 stars, 620 reviews).

Since the **Seaside Inn** has the highest number of reviews, I looked into dining options nearby. The local food critic highly recommends **Outerlands**, noting that it is the best spot in that neighborhood for a truly worthwhile sourdough. 

I hope you have a fantastic dinner! Let me know if you need any further assistance.""",
      thought_signature=b'\x12q\no\x01\x11M2\x0f\x18=\x1fc\xbcOE\x11\x05`\xef\xaa\xdf\\n\xc9Q\xc80Q\x03tlT\xf1\x86\xfa\x92\x88\x96\xfb\xab!c\xe8\xd9\xdfw$\xa9=Gw\xc7\xa8\xb0xw\xe9\xc6\x82\x0e0\x89\xd3\x03<\x81\x1e\x80\x1a\xf9\xdd\x85\r9\x86\xefL\xb7\x1a\xeco\xb8\xb0\xa4\x80\x18\x1d\xca?\x03\xd2\x9a\xe5<\xe2\x80...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None turn_complete_r

The top-rated hotels in San Francisco found in our database are **The Grand Hotel** (5 stars, 450 reviews) and the **Seaside Inn** (4 stars, 620 reviews).

Since the **Seaside Inn** has the highest number of reviews, I looked into dining options nearby. The local food critic highly recommends **Outerlands**, noting that it is the best spot in that neighborhood for a truly worthwhile sourdough. 

I hope you have a fantastic dinner! Let me know if you need any further assistance.

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [11]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-3.1-flash-lite",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [12]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: 1f4be726-e2db-4092-be39-98bfb1531662

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '1f4be726-e2db-4092-be39-98bfb1531662'...


Node execution failed with exception
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asynci

Root node multi_day_trip_agent failed.
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asyn

⏳ Rate limited by the API, retrying in 45s (attempt 1/6)...


Node execution failed with exception
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asynci

...

Node execution failed with exception
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asynci

Root node multi_day_trip_agent failed.
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asyn

⏳ Rate limited by the API, retrying in 45s (attempt 6/6)...



--------------------------------------------------
✅ Final Response:


--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [13]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: a55ea5a9-a056-4ba4-bd8f-4f9486f42d5f
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'a55ea5a9-a056-4ba4-bd8f-4f9486f42d5f'...


Node execution failed with exception
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asynci

Root node multi_day_trip_agent failed.
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asyn

⏳ Rate limited by the API, retrying in 45s (attempt 1/6)...


Node execution failed with exception
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asynci

...

Node execution failed with exception
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asynci

Root node multi_day_trip_agent failed.
Traceback (most recent call last):
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/adk/models/google_llm.py", line 322, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 8451, in generate_content
    response = await self._generate_content(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/models.py", line 6905, in _generate_content
    response = await self._api_client.async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1796, in async_request
    result = await self._async_request(
  File "/home/ubuntu/.local/lib/python3.10/site-packages/google/genai/_api_client.py", line 1729, in _async_request
    return await self._async_retry(  # type: ignore[no-any-return]
  File "/home/ubuntu/.local/lib/python3.10/site-packages/tenacity/asyn

⏳ Rate limited by the API, retrying in 45s (attempt 6/6)...



--------------------------------------------------
✅ Final Response:


--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
